excel to json and back

# Prepare and Define (DEFINITIONS + DATASET)

## config

In [8]:
verbose=1

In [9]:
#%mamba install -y openpyxl

## dataset

In [10]:
import openpyxl
print(openpyxl.__file__)
import pandas as pd
hb_df = pd.read_csv("strongs.csv")

/home/u/miniforge3/envs/py/lib/python3.12/site-packages/openpyxl/__init__.py


In [11]:
hb_df[:3]

,verse,word,form,form_en,strong_num,strong_title,strong_en_title,translit,translit_title,book,chapter
0,1,0,Παῦλος,Paul,3972,"3972: Paul, Paulus. Of Latin origin; Paulus, t...",N-NMS,Paulos,"Paulos: Paul, Paulus. Of Latin origin; Paulus,...",1_corinthians,1
1,1,1,κλητὸς,a called,2822,"2822: From the same as klesis; invited, i.e. A...",Adj-NMS,klētos,"klētos: From the same as klesis; invited, i.e....",1_corinthians,1
2,1,2,ἀπόστολος,apostle,652,"652: From apostello; a delegate; specially, an...",N-NMS,apostolos,apostolos: From apostello; a delegate; special...,1_corinthians,1


## define

In [12]:
"""
excel_json_converter.py
Converts between chapter JSON files and Excel (.xlsx) for review/editing.

JSON format (per verse):
  {
    "hebrew_words": [
      { "hebrew": "<hb_form>", "greek": [...], "latvian": [...] },
      ...
    ],
    "leftover_greek": [...],
    "leftover_latvian": [...]
  }

Excel format (9 columns):
  A: verse         (verse number, shown only on first row of each verse)
  B: form          (Hebrew word / LEFTOVER label)
  C: form_en       (English gloss from hb_df)
  D: latvian_1
  E: latvian_2
  F: latvian_3

Overflow: extra words get continuation rows with empty A/B.
Leftover rows use LEFTOVER {book} {chapter} {verse} {language} in col A,
with the same label on every continuation row.

Naming convention: excel/{book}/{chapter}.xlsx
"""

import json
import os
from pathlib import Path
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, PatternFill
from openpyxl.utils import get_column_letter

HEADERS = ["verse", "form", "form_en",
           "latvian_1", "latvian_2", "latvian_3"]

COL_WIDTHS = [6, 28, 22, 18, 18, 18]

HEADER_FILL = PatternFill("solid", start_color="D9E1F2")
LEFTOVER_FILL = PatternFill("solid", start_color="FFF2CC")
ALT_FILL = PatternFill("solid", start_color="F2F2F2")

RTL_FONT = Font(name="Arial", size=11)
LTR_FONT = Font(name="Arial", size=11)

from openpyxl.styles import Border, Side
_THIN = Side(style="thin")
VERSE_BORDER_BOTTOM = Border(bottom=_THIN)


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, max(len(lst), 1), n):
        yield lst[i:i + n]


def _leftover_label(book, chapter, verse, language):
    return f"LEFTOVER {book} {chapter} {verse} {language}"


def _get_hb_form_en(hb_df, book, chapter, verse, word_idx):
    """Return (form, form_en) from hb_df for a given word index in a verse."""
    try:
        verse_words = hb_df.query(
            f"book=='{book}' & chapter=={chapter} & verse=={verse}"
        ).sort_values("word")
        if word_idx < len(verse_words):
            row = verse_words.iloc[word_idx]
            return str(row["form"]), str(row["form_en"])
    except Exception:
        pass
    return "", ""


# ─────────────────────────────────────────────────────────────────────────────
# JSON → Excel
# ─────────────────────────────────────────────────────────────────────────────

def json_to_excel(json_path, hb_df, out_dir="excel"):
    """
    Read a chapter JSON file and write an Excel file.

    json_path : path like "bible/jeremiah/7.json"  or  "bible/jeremiah/7/"
                (if a directory is given, reads verse files 1.json, 2.json …)
    hb_df     : DataFrame with columns book, chapter, verse, word, form, form_en
    out_dir   : root output directory (default "excel")
    """
    # ── resolve book / chapter from path ──────────────────────────────────
    path = os.path.normpath(json_path)
    parts = path.replace("\\", "/").split("/")

    # Accept both "bible/jeremiah/7.json" and just "jeremiah/7.json" etc.
    # We need at least <book>/<chapter> at the end.
    if path.endswith(".json"):
        chapter_str = os.path.splitext(parts[-1])[0]
        book = parts[-2]
        data = _load_chapter_flat(json_path)
    else:
        # directory of per-verse files
        chapter_str = parts[-1]
        book = parts[-2]
        data = _load_chapter_dir(json_path)

    chapter = int(chapter_str)

    # ── build rows ────────────────────────────────────────────────────────
    # Each row: 9 values  [verse, form, form_en, gr1, gr2, gr3, lv1, lv2, lv3]
    rows = []
    fills = []       # per-row fill: None | "leftover"
    verse_ends = []  # 0-based row-list indices that are the last row of a verse

    for verse_idx, verse_data in enumerate(data):
        verse_num = verse_idx + 1
        hb_words = verse_data.get("greek_words", [])
        lo_latvian = verse_data.get("leftover_latvian", [])
        verse_start = len(rows)

        # ── mapped words ──────────────────────────────────────────────────
        for wi, word_entry in enumerate(hb_words):
            form, form_en = _get_hb_form_en(hb_df, book, chapter, verse_num, wi)
            latvian = word_entry.get("latvian", [])

            lv_chunks = list(_chunks(latvian, 3)) if latvian else [[]]
            n_rows = len(lv_chunks)

            for ri in range(n_rows):
                lv_c = lv_chunks[ri] if ri < len(lv_chunks) else []
                verse_col = verse_num if (wi == 0 and ri == 0) else ""
                form_col = form if ri == 0 else ""
                form_en_col = form_en if ri == 0 else ""
                rows.append([verse_col, form_col, form_en_col] + _pad3(lv_c))
                fills.append(None)

        # ── leftover latvian ──────────────────────────────────────────────
        if lo_latvian:
            label = _leftover_label(book, chapter, verse_num, "latvian")
            for lv_c in _chunks(lo_latvian, 3):
                rows.append(["", label, ""] + _pad3(lv_c))
                fills.append("leftover")

        # Mark last row of this verse for border drawing
        if len(rows) > verse_start:
            verse_ends.append(len(rows) - 1)

    # ── write workbook ────────────────────────────────────────────────────
    wb = Workbook()
    ws = wb.active
    ws.title = f"Chapter {chapter}"

    # Header row
    for ci, h in enumerate(HEADERS, 1):
        cell = ws.cell(row=1, column=ci, value=h)
        cell.font = Font(name="Arial", bold=True, size=11)
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center")

    ws.freeze_panes = "B2"

    verse_end_set = set(verse_ends)  # 0-based indices in rows[]

    # Data rows
    alt = False
    prev_verse = None
    for ri, (row, fill_type) in enumerate(zip(rows, fills), 2):
        row_verse = row[0]  # verse number or "" or leftover label
        # Toggle alt shading at the start of each new verse
        if row_verse and row_verse != prev_verse and not (isinstance(row_verse, str) and row_verse.startswith("LEFTOVER")):
            alt = not alt
            prev_verse = row_verse

        is_verse_end = (ri - 2) in verse_end_set  # convert to 0-based

        for ci, val in enumerate(row, 1):
            cell = ws.cell(row=ri, column=ci, value=val)
            cell.font = Font(name="Arial", size=11)
            cell.alignment = Alignment(
                horizontal="center" if ci == 1 else ("right" if ci == 3 else "left"),
                vertical="center"
            )
            if fill_type == "leftover":
                cell.fill = LEFTOVER_FILL
            elif alt:
                cell.fill = ALT_FILL
            if is_verse_end:
                cell.border = VERSE_BORDER_BOTTOM

    # Column widths
    for ci, w in enumerate(COL_WIDTHS, 1):
        ws.column_dimensions[get_column_letter(ci)].width = w

    # Save
    out_book_dir = os.path.join(out_dir, book)
    os.makedirs(out_book_dir, exist_ok=True)
    out_path = os.path.join(out_book_dir, f"{chapter}.xlsx")
    wb.save(out_path)
    if(verbose):
        print(f"✅ Exported {json_path} → {out_path}  ({len(rows)} rows, {len(verse_ends)} verses)")
    return out_path


def _pad3(lst):
    """Pad / trim a list to exactly 3 elements."""
    lst = list(lst)
    return (lst + ["", "", ""])[:3]


def _load_chapter_flat(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def _load_chapter_dir(dir_path):
    """Load per-verse json files 1.json, 2.json … in order."""
    verses = []
    i = 1
    while True:
        vpath = os.path.join(dir_path, f"{i}.json")
        if not os.path.exists(vpath):
            break
        with open(vpath, "r", encoding="utf-8") as f:
            verses.append(json.load(f))
        i += 1
    return verses


# ─────────────────────────────────────────────────────────────────────────────
# Excel → JSON
# ─────────────────────────────────────────────────────────────────────────────

def excel_to_json(xlsx_path, hb_df, out_dir="bible"):
    """
    Read an Excel chapter file and rebuild the chapter JSON.

    xlsx_path : path like "excel/jeremiah/7.xlsx"
    hb_df     : DataFrame with columns book, chapter, verse, word, form, form_en
    out_dir   : root output directory for JSON (default "bible")

    The function infers book and chapter from the file path.
    Output: out_dir/{book}/{chapter}.json  (flat chapter file)
    """
    path = os.path.normpath(xlsx_path)
    parts = path.replace("\\", "/").split("/")
    chapter_str = os.path.splitext(parts[-1])[0]
    book = parts[-2]
    chapter = int(chapter_str)

    # ── read sheet ────────────────────────────────────────────────────────
    wb = load_workbook(xlsx_path, data_only=True)
    ws = wb.active

    rows = []
    for row in ws.iter_rows(min_row=2, values_only=True):
        rows.append([str(v) if v is not None else "" for v in row])

    # ── build verse data using hb_df as the authority ─────────────────────
    verse_words = _get_chapter_hb_words(hb_df, book, chapter)

    # Pass 1: group excel rows into word-blocks
    # Columns: 0=verse, 1=form, 2=form_en, 3-5=greek, 6-8=latvian
    word_blocks = []
    current_block = None

    for row in rows:
        # Pad row to at least 6 cols
        row = (row + [""] * 6)[:6]
        form = row[1].strip()
        form_en = row[2].strip()
        lv = [v for v in row[3:6] if v]

        if form.startswith("LEFTOVER "):
            # Parse: LEFTOVER {book} {chapter} {verse} {language}
            parts_lo = form.split()
            # LEFTOVER <book> <chapter> <verse> <language>
            if len(parts_lo) >= 5:
                lo_book = parts_lo[1]
                lo_chap = int(parts_lo[2])
                lo_verse = int(parts_lo[3])
                lo_lang = parts_lo[4]
            else:
                lo_book, lo_chap, lo_verse, lo_lang = book, chapter, 1, "latvian"

            if current_block and current_block.get("leftover_type") == lo_lang \
                    and current_block.get("lo_info") == (lo_book, lo_chap, lo_verse, lo_lang):
                # continuation row for same leftover block
                current_block["latvian"].extend(lv)
            else:
                if current_block:
                    word_blocks.append(current_block)
                current_block = {
                    "form": form,
                    "form_en": "",
                    "latvian": list(lv),
                    "leftover_type": lo_lang,
                    "lo_info": (lo_book, lo_chap, lo_verse, lo_lang),
                }
        elif form:
            # New hebrew word row
            if current_block:
                word_blocks.append(current_block)
            current_block = {
                "form": form,
                "form_en": form_en,
                "latvian": list(lv),
                "leftover_type": None,
                "lo_info": None,
            }
        else:
            # Continuation row (col A empty)
            if current_block:
                current_block["latvian"].extend(lv)

    if current_block:
        word_blocks.append(current_block)

    # Pass 2: match word_blocks to verses using hb_df
    # For non-leftover blocks, we consume verse_words in order to assign verse/word index.
    # For leftover blocks, we use their embedded verse info.

    # Group by verse
    verse_map = {}   # verse_num → { 'hebrew_words': [...], 'leftover_greek': [...], 'leftover_latvian': [...] }
    vw_iter = iter(verse_words)
    current_vw = None

    def next_vw():
        nonlocal current_vw
        try:
            current_vw = next(vw_iter)
        except StopIteration:
            current_vw = None

    next_vw()

    for block in word_blocks:
        if block["leftover_type"] is not None:
            lo_book, lo_chap, lo_verse, lo_lang = block["lo_info"]
            vd = verse_map.setdefault(lo_verse, _empty_verse())
            vd["leftover_latvian"].extend(block["latvian"])
        else:
            if current_vw is None:
                print(f"  ⚠️  More word blocks than strongs_df words — stopping at: {block['form']}")
                break
            vn, wi, ref_form, ref_form_en = current_vw
            vd = verse_map.setdefault(vn, _empty_verse())
            vd["greek_words"].append({
                "greek": ref_form,  # always use authoritative greek form
                "latvian": block["latvian"],
            })
            next_vw()

    # Pass 3: build ordered verse list
    max_verse = max(verse_map.keys()) if verse_map else 0
    data = []
    for vn in range(1, max_verse + 1):
        if vn in verse_map:
            data.append(verse_map[vn])
        else:
            # Verse with no data — unlikely but safe
            data.append(_empty_verse())

    # ── serialize compactly (same format as original) ──────────────────────
    def word_to_compact(w):
        greek = w.get("greek", "")
        latvian = w.get("latvian", [])
        lv_str = json.dumps(latvian, ensure_ascii=False)
        return f'{{ "greek": "{greek}", "latvian": {lv_str} }}'

    verse_blocks = []
    for verse in data:
        words = verse.get("greek_words", [])
        compact_words = ",\n      ".join(word_to_compact(w) for w in words)
        lo_lv = json.dumps(verse.get("leftover_latvian", []), ensure_ascii=False)
        verse_blocks.append(
            f'  {{\n    "greek_words": [\n      {compact_words}\n    ],\n'
            f'    "leftover_latvian": {lo_lv}\n  }}'
        )

    new_raw = "[\n" + ",\n".join(verse_blocks) + "\n]"

    out_book_dir = os.path.join(out_dir, book)
    os.makedirs(out_book_dir, exist_ok=True)
    out_path = os.path.join(out_book_dir, f"{chapter}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(new_raw)

    if(verbose):
        print(f"✅ Imported {xlsx_path} → {out_path}  ({max_verse} verses)")
    return out_path


def _empty_verse():
    return {"greek_words": [], "leftover_latvian": []}


def _get_chapter_hb_words(hb_df, book, chapter):
    """Return ordered list of (verse, word_idx, form, form_en) for a chapter."""
    ch = hb_df.query(f"book=='{book}' & chapter=={chapter}").copy()
    ch = ch.sort_values(["verse", "word"])
    result = []
    for _, row in ch.iterrows():
        result.append((int(row["verse"]), int(row["word"]), str(row["form"]), str(row["form_en"])))
    return result


# ─────────────────────────────────────────────────────────────────────────────
# Convenience: export / import whole book
# ─────────────────────────────────────────────────────────────────────────────

def export_book(book, hb_df, json_root="bible", excel_root="excel"):
    """Export all chapter JSON files for a book to Excel."""
    book_dir = os.path.join(json_root, book)
    if not os.path.isdir(book_dir):
        print(f"  ⚠️  Book directory not found: {book_dir}")
        return
    for fname in sorted(os.listdir(book_dir), key=lambda x: int(os.path.splitext(x)[0]) if os.path.splitext(x)[0].isdigit() else 0):
        if fname.endswith(".json") and os.path.splitext(fname)[0].isdigit():
            json_to_excel(os.path.join(book_dir, fname), hb_df, out_dir=excel_root)


def import_book(book, hb_df, excel_root="excel", json_root="bible"):
    """Import all chapter Excel files for a book back to JSON."""
    book_dir = os.path.join(excel_root, book)
    if not os.path.isdir(book_dir):
        print(f"  ⚠️  Excel book directory not found: {book_dir}")
        return
    for fname in sorted(os.listdir(book_dir), key=lambda x: int(os.path.splitext(x)[0]) if os.path.splitext(x)[0].isdigit() else 0):
        if fname.endswith(".xlsx") and os.path.splitext(fname)[0].isdigit():
            excel_to_json(os.path.join(book_dir, fname), hb_df, out_dir=json_root)

In [13]:
import subprocess
import sys
SERVER_ROOT_BIBLE_EXCEL_FOLDER="/var/www/noit/th/e/excel/"
SERVER_LOGIN_PATH="root@noit.pro"
LOG_SCP_OUTPUT=1
def upload_one_excel_chap(book, curchap, excel_root="excel"):
        print(f'uploading.. [{str(Path(excel_root) / str(book) / f"{curchap}.xlsx")}]')
        r = subprocess.run(["scp", str(Path(excel_root) / str(book) / f"{curchap}.xlsx"),
                            f'{SERVER_LOGIN_PATH}:{SERVER_ROOT_BIBLE_EXCEL_FOLDER}{str(book)}/'],
            #capture_output=True, 
            #text=True)                          
                          )
        if(LOG_SCP_OUTPUT):
            #print("scp exit code:", r.returncode)
            if r.returncode == 0:
                print(f'🚀 successfully uploaded {str(Path(excel_root) / str(book) / f"{curchap}.xlsx")}')
            else:
                print(f"error uploading, exit code: {r.returncode}")
    

In [14]:
def run_zip_excel_folder_on_server(archive_name="excel.zip", exclude_folder_list=[]):
    print("zipping folder")
    exclude_args = " ".join(f'"*/{x}/*"' for x in exclude_folder_list)
    if(not exclude_folder_list):
        remote_cmd = f"cd {SERVER_ROOT_BIBLE_EXCEL_FOLDER} && zip -q {archive_name} -r . && du -sh {archive_name}"
    else:
    #remote_cmd = f"cd {SERVER_ROOT_BIBLE_FOLDER} && zip -q {archive_name} -r . -x {exclude_args} && ls -lh {archive_name}"
        remote_cmd = f"cd {SERVER_ROOT_BIBLE_EXCEL_FOLDER} && zip -q {archive_name} -r . -x {exclude_args} && du -sh {archive_name}"
    rr = subprocess.run(["ssh",
                        f'{SERVER_LOGIN_PATH}',
                         remote_cmd]
                       )
    if(rr.returncode !=0):
        print(f"return code from zipping: {rr.returncode}")

# EXPORT ALL

In [16]:
import os
from pathlib import Path
from tqdm.notebook import tqdm
jsonroot="bible"
excelroot="excel"
VERBOSE_BAK1 = verbose 
verbose = False
books = [d.name for d in Path(jsonroot).iterdir() if d.is_dir() and not d.name==excelroot]

for book in tqdm(sorted(books), desc=f"⚡generate xlsx"):
    book_dir = os.path.join(jsonroot, book)
    if not os.path.isdir(book_dir):
        print(f"  ⚠️  Book directory not found: {book_dir}")  
        continue
    for fname in tqdm(sorted([f for f in Path(book_dir).glob("*.json") if f.stem.isdigit()], key=lambda x: int(x.stem) if x.stem.isdigit() else 0),
                      leave=False, desc=f"⚡ {book}"):
        json_to_excel(fname, hb_df, out_dir=excelroot)  
verbose = VERBOSE_BAK1

⚡generate xlsx:   0%|          | 0/27 [00:00<?, ?it/s]

⚡ 1_corinthians:   0%|          | 0/16 [00:00<?, ?it/s]

⚡ 1_john:   0%|          | 0/5 [00:00<?, ?it/s]

⚡ 1_peter:   0%|          | 0/5 [00:00<?, ?it/s]

⚡ 1_thessalonians:   0%|          | 0/5 [00:00<?, ?it/s]

...

⚡ philemon:   0%|          | 0/1 [00:00<?, ?it/s]

⚡ philippians:   0%|          | 0/4 [00:00<?, ?it/s]

⚡ revelation:   0%|          | 0/22 [00:00<?, ?it/s]

⚡ romans:   0%|          | 0/16 [00:00<?, ?it/s]

⚡ titus:   0%|          | 0/3 [00:00<?, ?it/s]

In [21]:
def import_excel_to_json(book, chapter):
    excel_to_json(os.path.join(os.path.join("excel", book), f"{chapter}.xlsx"), hb_df)
import_excel_to_json("1_corinthians", 1)

✅ Imported excel/1_corinthians/1.xlsx → bible/1_corinthians/1.json  (31 verses)


# old runner 2

In [26]:
def import_excel_to_json(book, chapter):
    excel_to_json(os.path.join(os.path.join("excel", book), f"{chapter}.xlsx"), hb_df)
def export_json_to_excel(book, chapter):
    json_to_excel(os.path.join("bible", book, f"{chapter}.json"), hb_df)
def r():
    return range(ii, ii+3)

DO_UPLOAD=1
LOG_SCP_OUTPUT=1
book="jeremiah"
book="1_chronicles"
book="proverbs"
book="psalms"
book="songs"
book="ruth"
book="1_kings"
book="job"
book="ezekiel"
ii=46
#r = range(1, 51)
#r = range(1, 67)
#r = range(1, 7)
skip_list = []
for i in r():
#    import_excel_to_json(book, i)
     export_json_to_excel(book, i)
if(DO_UPLOAD):
    for i in r():
        upload_one_excel_chap(book, i)
    run_zip_excel_folder_on_server()

✅ Exported bible/ezekiel/46.json → excel/ezekiel/46.xlsx  (501 rows, 24 verses)
✅ Exported bible/ezekiel/47.json → excel/ezekiel/47.xlsx  (508 rows, 23 verses)
✅ Exported bible/ezekiel/48.json → excel/ezekiel/48.xlsx  (724 rows, 35 verses)
uploading.. [excel/ezekiel/46.xlsx]
🚀 successfully uploaded excel/ezekiel/46.xlsx
uploading.. [excel/ezekiel/47.xlsx]
🚀 successfully uploaded excel/ezekiel/47.xlsx
uploading.. [excel/ezekiel/48.xlsx]
🚀 successfully uploaded excel/ezekiel/48.xlsx
zipping folder
12M	excel.zip


# current run

In [32]:
def import_excel_to_json(book, chapter):
    excel_to_json(os.path.join(os.path.join("excel", book), f"{chapter}.xlsx"), hb_df)
def export_json_to_excel(book, chapter):
    json_to_excel(os.path.join("bible", book, f"{chapter}.json"), hb_df)
def r():
    return range(ii, ii+5)

DO_UPLOAD=1
LOG_SCP_OUTPUT=1
book="exodus"
book="numbers"
book="2_chronicles"
book="deuteronomy"
book="1_samuel"
ii=1

skip_list = []
for i in r():
     export_json_to_excel(book, i)
if(DO_UPLOAD):
    for i in r():
        upload_one_excel_chap(book, i)
    run_zip_excel_folder_on_server()

✅ Exported bible/1_samuel/1.json → excel/1_samuel/1.xlsx  (546 rows, 28 verses)
✅ Exported bible/1_samuel/2.json → excel/1_samuel/2.xlsx  (720 rows, 36 verses)
✅ Exported bible/1_samuel/3.json → excel/1_samuel/3.xlsx  (376 rows, 21 verses)
✅ Exported bible/1_samuel/4.json → excel/1_samuel/4.xlsx  (478 rows, 22 verses)
✅ Exported bible/1_samuel/5.json → excel/1_samuel/5.xlsx  (298 rows, 12 verses)
uploading.. [excel/1_samuel/1.xlsx]
🚀 successfully uploaded excel/1_samuel/1.xlsx
uploading.. [excel/1_samuel/2.xlsx]
🚀 successfully uploaded excel/1_samuel/2.xlsx
uploading.. [excel/1_samuel/3.xlsx]
🚀 successfully uploaded excel/1_samuel/3.xlsx
uploading.. [excel/1_samuel/4.xlsx]
🚀 successfully uploaded excel/1_samuel/4.xlsx
uploading.. [excel/1_samuel/5.xlsx]
🚀 successfully uploaded excel/1_samuel/5.xlsx
zipping folder
17M	excel.zip
